In [ ]:
!kaggle datasets download -d trainingdatapro/real-vs-fake-anti-spoofing-video-classification

Dataset URL: https://www.kaggle.com/datasets/trainingdatapro/real-vs-fake-anti-spoofing-video-classification
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100% 3.04G/3.04G [00:36<00:00, 89.0MB/s]



In [ ]:
!unzip -q real-vs-fake-anti-spoofing-video-classification.zip -d dados_simulados/

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import os

# 1. Mapeamento das Pastas (Data Loader)
# O Keras lerá as subpastas automaticamente e atribuirá as classes (Real = 0, Fake/Spoof = 1)
caminho_dados = 'dataset_imagens/'

print("Carregando o lote de dados para treinamento...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    caminho_dados,
    validation_split=0.2, # Separa 20% das imagens para testar o modelo
    subset="training",
    seed=123,
    image_size=(224, 224), # Padrão exigido pela MobileNetV2
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    caminho_dados,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32
)

# 2. Importando a Arquitetura de Alto Desempenho (Transfer Learning)
print("Construindo a Rede Neural...")
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Congelamos o conhecimento prévio para o MVP rodar mais rápido

# 3. Adicionando a nossa camada de Liveness Detection
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
# A saída será um número entre 0 e 1 (Probabilidade de ser Fraude)
predicoes = Dense(1, activation='sigmoid')(x)

modelo = Model(inputs=base_model.input, outputs=predicoes)

# 4. Compilação do Modelo
modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Treinamento
print("Iniciando o treinamento (Aguarde a conclusão das 3 épocas)...")
historia = modelo.fit(train_ds, validation_data=val_ds, epochs=3)

# 6. Exportação do Arquivo Final
modelo.save('modelo_liveness.h5')
print("Treinamento finalizado! Arquivo 'modelo_liveness.h5' salvo com sucesso.")

Carregando o lote de dados para treinamento...
Found 480 files belonging to 2 classes.
Using 384 files for training.
Found 480 files belonging to 2 classes.
Using 96 files for validation.
Construindo a Rede Neural...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Iniciando o treinamento (Aguarde a conclusão das 3 épocas)...
Epoch 1/3
12/12 ━━━━━━━━━━━━━━━━━━━━ 33s 2s/step - accuracy: 0.6927 - loss: 0.5812 - val_accuracy: 0.7812 - val_loss: 0.4797
Epoch 2/3
12/12 ━━━━━━━━━━━━━━━━━━━━ 37s 2s/step - accuracy: 0.8490 - loss: 0.3642 - val_accuracy: 0.8125 - val_loss: 0.4830
Epoch 3/3
12/12 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.8672 - loss: 0.3075 - val_accuracy: 0.7917 - val_loss: 0.5037


Treinamento finalizado! Arquivo 'modelo_liveness.h5' salvo com sucesso.


In [ ]:
import glob

# Busca todos os arquivos dentro da pasta, independentemente da extensão
arquivos = glob.glob('dados_simulados/**/*.*', recursive=True)

print(f"Total de arquivos encontrados na nuvem: {len(arquivos)}")
print("Amostra dos 5 primeiros arquivos (para entendermos a estrutura das pastas):")
for arq in arquivos[:5]:
    print(arq)

Total de arquivos encontrados na nuvem: 161
Amostra dos 5 primeiros arquivos (para entendermos a estrutura das pastas):
dados_simulados/real_and_fake.csv
dados_simulados/train/real_video/0.mp4
dados_simulados/train/real_video/15.mp4
dados_simulados/train/real_video/14.mp4
dados_simulados/train/real_video/2.mp4


In [ ]:
import cv2
import os
import glob

# 1. Criação da nova estrutura de pastas (Data Lake simulado)
pasta_destino_real = 'dataset_imagens/real'
pasta_destino_spoof = 'dataset_imagens/spoof'
os.makedirs(pasta_destino_real, exist_ok=True)
os.makedirs(pasta_destino_spoof, exist_ok=True)

print("Iniciando a esteira de extração de frames...")

# 2. Mapeamento dos arquivos de vídeo
videos = glob.glob('dados_simulados/**/*.mp4', recursive=True)
frames_por_video = 3 # Amostragem para o MVP ser leve e rápido

# 3. Processamento e conversão (Engenharia de Dados)
for caminho_video in videos:
    # Classifica dinamicamente lendo o nome da pasta de origem
    if 'real' in caminho_video.lower():
        pasta_salvar = pasta_destino_real
        prefixo = 'real_'
    else:
        pasta_salvar = pasta_destino_spoof
        prefixo = 'spoof_'

    nome_base = os.path.basename(caminho_video).replace('.mp4', '')
    captura = cv2.VideoCapture(caminho_video)

    # Lógica matemática para capturar fotos espaçadas do vídeo
    total_frames = int(captura.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        continue

    passo = max(1, total_frames // frames_por_video)

    contador_fotos = 0
    for i in range(0, total_frames, passo):
        if contador_fotos >= frames_por_video:
            break

        captura.set(cv2.CAP_PROP_POS_FRAMES, i)
        sucesso, frame = captura.read()

        if sucesso:
            # Salva o frame convertido em JPG de alta qualidade
            nome_arquivo = f"{pasta_salvar}/{prefixo}{nome_base}_frame{contador_fotos}.jpg"
            cv2.imwrite(nome_arquivo, frame)
            contador_fotos += 1

    captura.release()

print(f"Extração concluída com sucesso! Novo dataset estruturado em: 'dataset_imagens/'")

Iniciando a esteira de extração de frames...
Extração concluída com sucesso! Novo dataset estruturado em: 'dataset_imagens/'
